# Importing county population growth data

Note: This script comes from census_data_imports.ipynb within my Python for Nonprofits guide. For more information on how this code works, visit https://github.com/kburchfiel/pfn/blob/main/Census_Data_Imports/census_data_imports.ipynb .

In [1]:
import time
program_start_time = time.time()
import pandas as pd
pd.set_option('display.max_columns', 1000) # This max column setting
# will prevent columns in the notebook from being hidden; however,
# if render_for_pdf (defined in the Appendix's helper_funcs.py file)
# is set to True, this setting will get overwritten.
import numpy as np
from iteration_utilities import duplicates

import sys
sys.path.insert(1, '../Appendix')

acs5_year = 2024

download_new_variable_list = False

In [2]:
with open ('census_api_key_path.txt') as file:
    key_path = file.read()
with open(key_path) as file:
    key = file.read()

In [3]:
from census_import_scripts import download_variable_list, \
create_variable_aliases, retrieve_census_data, create_comparison_fields

## Retrieving variable and group information

In [4]:
if download_new_variable_list == True: # This process can take a little
    # while, so if you already have a copy of the variable list you need,
    # consider setting download_variable_list to False.
    download_variable_list(acs5_year, 'acs5')

# Reading the group and variable datasets into our script:
df_variables = pd.read_csv(
    f'Datasets/acs5_{acs5_year}_variables.csv')

df_groups = pd.read_csv(
    f'Datasets/acs5_{acs5_year}_groups.csv')

In [5]:
df_variables.head()

,Name,Label,Concept,Required,Attributes,Limit,Predicate Type,Group
0,B01001_001E,Estimate!!Total:,Sex by Age,not required,"B01001_001EA, B01001_001M, B01001_001MA",0,int,B01001
1,B01001_002E,Estimate!!Total:!!Male:,Sex by Age,not required,"B01001_002EA, B01001_002M, B01001_002MA",0,int,B01001
2,B01001_003E,Estimate!!Total:!!Male:!!Under 5 years,Sex by Age,not required,"B01001_003EA, B01001_003M, B01001_003MA",0,int,B01001
3,B01001_004E,Estimate!!Total:!!Male:!!5 to 9 years,Sex by Age,not required,"B01001_004EA, B01001_004M, B01001_004MA",0,int,B01001
4,B01001_005E,Estimate!!Total:!!Male:!!10 to 14 years,Sex by Age,not required,"B01001_005EA, B01001_005M, B01001_005MA",0,int,B01001


In [6]:
df_groups.head()

,Concept,Group
0,Sex by Age,B01001
1,Sex by Age (White Alone),B01001A
2,Sex by Age (Black or African American Alone),B01001B
3,Sex by Age (American Indian and Alaska Native ...,B01001C
4,Sex by Age (Asian Alone),B01001D


In [7]:
variable_list = [
    'B01001_001E'] # Total population

## Creating aliases and specifying retrieval years

Creating our aliases:

In [8]:
alias_dict = create_variable_aliases(
    df_variables=df_variables, 
    variable_list=variable_list)
alias_dict

{'B01001_001E': 'Sex by Age_Estimate!!Total: (B01001_001E)'}

In [9]:
# Note: 2009 appears to be the earliest year for which ACS5 data 
# is available (at least through the API).
years_to_retrieve = [acs5_year - 15,
                     acs5_year - 10,
                     acs5_year - 5,
                     acs5_year]
years_to_retrieve

[2009, 2014, 2019, 2024]

At this point, it's a good idea to confirm that our three variable codes ('B01001_001E', 'B01001_011E', and 'B01001_035E') had the same meaning for all the years whose data we'll be retrieving. We can do so by running the following code, which retrieves these variables and their corresponding descriptions for all of the years in years_to_retrieve.

(Due to the size of the variables.html page, this code can take a while to run, so I commented it out below.)

In [10]:
var_meanings_by_year_df_list = []
for year in years_to_retrieve:
    df_var_list = pd.read_html(
        f'https://api.census.gov/data/{year}/acs/acs5/variables.html')[
    0][['Name', 'Label', 'Concept']].query(
        "Name in @variable_list")
    df_var_list.insert(0, 'Year', year)
    var_meanings_by_year_df_list.append(df_var_list)
df_var_meanings_by_year = pd.concat(
    [df for df in var_meanings_by_year_df_list])
df_var_meanings_by_year.to_csv('var_meanings_by_year.csv', index=False)
df_var_meanings_by_year

,Year,Name,Label,Concept
6,2009,B01001_001E,Estimate!!Total,SEX BY AGE
6,2014,B01001_001E,Estimate!!Total,SEX BY AGE
4,2019,B01001_001E,Estimate!!Total:,SEX BY AGE
4,2024,B01001_001E,Estimate!!Total:,Sex by Age


Here's a series of outputs from the saved .csv copy of this table that confirms that these codes had the same meaning in each of the years we're analyzing:

In [11]:
df_var_meanings_by_year = pd.read_csv(
    'var_meanings_by_year.csv')
for name in variable_list:
    print(df_var_meanings_by_year.query("Name == @name"))

   Year         Name             Label     Concept
0  2009  B01001_001E   Estimate!!Total  SEX BY AGE
1  2014  B01001_001E   Estimate!!Total  SEX BY AGE
2  2019  B01001_001E  Estimate!!Total:  SEX BY AGE
3  2024  B01001_001E  Estimate!!Total:  Sex by Age


In [12]:
census_data_by_year_df_list = []
for year in years_to_retrieve:
    print("Retrieving data for", year)
    df_data = retrieve_census_data(
        survey='acs5', year=year, 
        region='county',
        variable_list=variable_list, 
        rename_data_fields=True, 
        field_vars_dict=alias_dict, key=key)
    census_data_by_year_df_list.append(df_data)
df_growth_data_by_year = pd.concat(df for df in census_data_by_year_df_list)
# Removing Puerto Rico from our list of results so as to focus only on counties
# and county equivalents within the 50 US states and DC:
df_growth_data_by_year.query("state != '72'", inplace=True)
df_growth_data_by_year

Retrieving data for 2009
Retrieving data from: https://api.census.gov/data/2009/acs/acs5?get=NAME,B01001_001E&for=county:*&key=eac3512bdfd84bf4a86f2051d6298bb57d1bbc9d
Retrieving data for 2014
Retrieving data from: https://api.census.gov/data/2014/acs/acs5?get=NAME,B01001_001E&for=county:*&key=eac3512bdfd84bf4a86f2051d6298bb57d1bbc9d
Retrieving data for 2019
Retrieving data from: https://api.census.gov/data/2019/acs/acs5?get=NAME,B01001_001E&for=county:*&key=eac3512bdfd84bf4a86f2051d6298bb57d1bbc9d
Retrieving data for 2024
Retrieving data from: https://api.census.gov/data/2024/acs/acs5?get=NAME,B01001_001E&for=county:*&key=eac3512bdfd84bf4a86f2051d6298bb57d1bbc9d


,Year,county,NAME,state,Sex by Age_Estimate!!Total: (B01001_001E)
1,2009,091,"Dodge County, Georgia",13,19695
2,2009,093,"Dooly County, Georgia",13,11641
3,2009,095,"Dougherty County, Georgia",13,95330
4,2009,097,"Douglas County, Georgia",13,122657
5,2009,099,"Early County, Georgia",13,11812
...,...,...,...,...,...
3140,2024,037,"Sweetwater County, Wyoming",56,41542
3141,2024,039,"Teton County, Wyoming",56,23396
3142,2024,041,"Uinta County, Wyoming",56,20644
3143,2024,043,"Washakie County, Wyoming",56,7703


In [13]:
df_growth_data_by_year.rename(
    columns={'Sex by Age_Estimate!!Total: (B01001_001E)':'Total_Pop'}, 
inplace=True)
df_growth_data_by_year

,Year,county,NAME,state,Total_Pop
1,2009,091,"Dodge County, Georgia",13,19695
2,2009,093,"Dooly County, Georgia",13,11641
3,2009,095,"Dougherty County, Georgia",13,95330
4,2009,097,"Douglas County, Georgia",13,122657
5,2009,099,"Early County, Georgia",13,11812
...,...,...,...,...,...
3140,2024,037,"Sweetwater County, Wyoming",56,41542
3141,2024,039,"Teton County, Wyoming",56,23396
3142,2024,041,"Uinta County, Wyoming",56,20644
3143,2024,043,"Washakie County, Wyoming",56,7703


In [14]:
df_growth_data_by_year_pivot = df_growth_data_by_year.copy().pivot(
    columns='Year', index=['NAME', 'county', 'state']).reset_index()
# The values could be named explicitly, but since pivot() will infer them
    # automatically, there's no need to do so. I thus removed the following
# code from the pivot() call:
    # values=['Total_Pop',
    #           'Total_Pop_25_to_29']

df_growth_data_by_year_pivot.head()

0                                 NAME county state Total_Pop            \
Year                                                     2009      2014   
0     Abbeville County, South Carolina    001    45   25347.0   25100.0   
1             Acadia Parish, Louisiana    001    22   59616.0   62031.0   
2            Accomack County, Virginia    001    51   38522.0   33165.0   
3                    Ada County, Idaho    001    16  368791.0  409239.0   
4                   Adair County, Iowa    001    19    7555.0    7543.0   

0                         
Year      2019      2024  
0      24627.0   24420.0  
1      62457.0   56955.0  
2      32673.0   33335.0  
3     456849.0  518935.0  
4       7085.0    7460.0

In [15]:
df_growth_data_by_year_pivot.columns = (
    df_growth_data_by_year_pivot.columns.to_flat_index())

df_growth_data_by_year_pivot.head()

,"(NAME, )","(county, )","(state, )","(Total_Pop, 2009)","(Total_Pop, 2014)","(Total_Pop, 2019)","(Total_Pop, 2024)"
0,"Abbeville County, South Carolina",001,45,25347.0,25100.0,24627.0,24420.0
1,"Acadia Parish, Louisiana",001,22,59616.0,62031.0,62457.0,56955.0
2,"Accomack County, Virginia",001,51,38522.0,33165.0,32673.0,33335.0
3,"Ada County, Idaho",001,16,368791.0,409239.0,456849.0,518935.0
4,"Adair County, Iowa",001,19,7555.0,7543.0,7085.0,7460.0


In [16]:
df_growth_data_by_year_pivot.columns = [
    column[0] + '_' + str(column[1]) if isinstance(column[1], int) 
    else column[0] for column in df_growth_data_by_year_pivot.columns]
df_growth_data_by_year_pivot.head()

,NAME,county,state,Total_Pop_2009,Total_Pop_2014,Total_Pop_2019,Total_Pop_2024
0,"Abbeville County, South Carolina",001,45,25347.0,25100.0,24627.0,24420.0
1,"Acadia Parish, Louisiana",001,22,59616.0,62031.0,62457.0,56955.0
2,"Accomack County, Virginia",001,51,38522.0,33165.0,32673.0,33335.0
3,"Ada County, Idaho",001,16,368791.0,409239.0,456849.0,518935.0
4,"Adair County, Iowa",001,19,7555.0,7543.0,7085.0,7460.0


In [17]:
df_growth_data_by_year_pivot['GEOID'] = df_growth_data_by_year_pivot[
'state'].astype('str').str.zfill(2) + df_growth_data_by_year_pivot['county'].astype('str').str.zfill(3)
df_growth_data_by_year_pivot

,NAME,county,state,Total_Pop_2009,Total_Pop_2014,Total_Pop_2019,Total_Pop_2024,GEOID
0,"Abbeville County, South Carolina",001,45,25347.0,25100.0,24627.0,24420.0,45001
1,"Acadia Parish, Louisiana",001,22,59616.0,62031.0,62457.0,56955.0,22001
2,"Accomack County, Virginia",001,51,38522.0,33165.0,32673.0,33335.0,51001
3,"Ada County, Idaho",001,16,368791.0,409239.0,456849.0,518935.0,16001
4,"Adair County, Iowa",001,19,7555.0,7543.0,7085.0,7460.0,19001
...,...,...,...,...,...,...,...,...
3154,"Yuma County, Arizona",027,04,188983.0,201453.0,209468.0,211741.0,04027
3155,"Yuma County, Colorado",125,08,9630.0,10131.0,10003.0,9979.0,08125
3156,"Zapata County, Texas",505,48,13561.0,14231.0,14304.0,13841.0,48505
3157,"Zavala County, Texas",507,48,11620.0,12013.0,12039.0,9412.0,48507


Data for Connecticut counties won't be comparable across a number of years due to changes to the county definitions in the early 2020s:

In [18]:
df_growth_data_by_year_pivot.query("NAME.str.contains('Connecticut')")

,NAME,county,state,Total_Pop_2009,Total_Pop_2014,Total_Pop_2019,Total_Pop_2024,GEOID
385,"Capitol Planning Region, Connecticut",110,09,NaN,NaN,NaN,977290.0,09110
903,"Fairfield County, Connecticut",001,09,892843.0,934215.0,943926.0,NaN,09001
1107,"Greater Bridgeport Planning Region, Connecticut",120,09,NaN,NaN,NaN,329259.0,09120
1227,"Hartford County, Connecticut",003,09,874409.0,897374.0,893561.0,NaN,09003
1696,"Litchfield County, Connecticut",005,09,188411.0,187542.0,182002.0,NaN,09005
1728,Lower Connecticut River Valley Planning Region...,130,09,NaN,NaN,NaN,175822.0,09130
1919,"Middlesex County, Connecticut",007,09,164004.0,165534.0,163053.0,NaN,09007
2053,"Naugatuck Valley Planning Region, Connecticut",140,09,NaN,NaN,NaN,454969.0,09140
2068,"New Haven County, Connecticut",009,09,843532.0,863148.0,857513.0,NaN,09009
2070,"New London County, Connecticut",011,09,266183.0,274071.0,267390.0,NaN,09011


In [ ]:
df_growth_data_by_year_pivot_condensed = df_growth_data_by_year_pivot.drop(['NAME', 'county', 'state'], axis = 1).copy()
df_growth_data_by_year_pivot_condensed

The following code displays this string in a format that will be easier to import into D3:

In [ ]:
df_growth_data_by_year_pivot_condensed.to_csv('Datasets/county_growth_data.csv', index = False)

In [ ]:
df_growth_data_by_year_pivot_condensed.to_csv(index = False)